In [1]:
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from datasets.arrow_dataset import Dataset as HFDataset

from torchvision.transforms import v2

from torch import Tensor
from torch.utils.data import Dataset, DataLoader

from torch.profiler import (
    profile,
    ProfilerActivity,
    schedule,
    tensorboard_trace_handler,
    record_function,
)

from tqdm import tqdm

from vit import *
from train import *
from eval import *

from IPython.display import clear_output

import sys
import time

import warnings

warnings.filterwarnings("ignore", message="Corrupt EXIF data.*", category=UserWarning)
warnings.filterwarnings("ignore", message="Metadata Warning*", category=UserWarning)
warnings.filterwarnings("ignore", message="Truncated File Read*", category=UserWarning)


train_ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="train",
)

val_ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="validation",
)

test_ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="test",
)

num_train_samples = len(train_ds)
num_val_samples = len(val_ds)
num_test_samples = len(test_ds)

profiler_schedule = schedule(wait=10, warmup=2, active=8, repeat=1)


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/267 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

... (4 outputs left out)

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

In [2]:
torch.manual_seed(42)

class ImageNetDataset(Dataset):
    def __init__(self, dataset: HFDataset, transform: v2.Transform = None) -> None:
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx: int) -> tuple[Tensor, int]:
        sample = self.dataset[idx]

        image = sample["image"]
        if image.mode != "RGB":
            image = image.convert("RGB") # Don't want RGBA

        label = sample["label"]

        if self.transform is not None:
            image = self.transform(image)

        return image, label


        # t0 = time.perf_counter()
        # sample = self.dataset[idx]
        # t1 = time.perf_counter()

        # image = sample["image"]
        # label = sample["label"]

        # if image.mode != "RGB":
        #     image = image.convert("RGB")

        # t2 = time.perf_counter()

        # if self.transform is not None:
        #     image = self.transform(image)

        # t3 = time.perf_counter()

        # return image, label, t1 - t0, t2 - t1, t3 - t2

eval_transform = v2.Compose([
    v2.ToImage(),

    # preserves aspect ratio, addresses varying image resolutions
    v2.Resize(size=256, interpolation=v2.InterpolationMode.BICUBIC),
    v2.CenterCrop(size=224),

    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

train_transform = v2.Compose([
    v2.ToImage(),

    v2.RandomResizedCrop( # defaults
        size=224,
        scale=(0.08, 1.0),
        ratio=(3/4, 4/3),
        interpolation=v2.InterpolationMode.BICUBIC
    ),
    v2.RandomHorizontalFlip(p=0.5),

    # v2.ToDtype(torch.float32, scale=True),
    # v2.Normalize(
    #     mean=[0.485, 0.456, 0.406], 
    #     std=[0.229, 0.224, 0.225]
    # ),
])

mixup = v2.MixUp(alpha=1.0, num_classes=1000)

clean_train_dataset = ImageNetDataset(train_ds, eval_transform)
train_dataset = ImageNetDataset(train_ds, train_transform)

val_dataset = ImageNetDataset(val_ds, eval_transform)
test_dataset = ImageNetDataset(test_ds, eval_transform)


effective_batch_size = 4096
micro_batch_size = 256

assert effective_batch_size % micro_batch_size == 0, \
    f"micro-batch size {micro_batch_size} must divide effective batch size {effective_batch_size}"
grad_accum_steps = effective_batch_size // micro_batch_size

num_workers = 20

train_dataloader = DataLoader(
    train_dataset,
    batch_size = micro_batch_size,
    shuffle = False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    # prefetch_factor=4
)

clean_train_dataloader = DataLoader(
    clean_train_dataset,
    batch_size = micro_batch_size,
    shuffle = False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    # prefetch_factor=4
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size = micro_batch_size,
    shuffle = False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    # prefetch_factor=4
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size = micro_batch_size,
    shuffle = False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    # prefetch_factor=4
)


# times = []

# # Let workers fill their prefetch queues
# for i in range(10):
#     loader_iter = iter(train_dataloader)

#     for _ in range(30):
#         next(loader_iter)

#     num_batches = 300

#     start = time.perf_counter()

#     for _ in range(num_batches):
#         next(loader_iter)

#     elapsed = time.perf_counter() - start
#     times.append(num_batches * micro_batch_size / elapsed)


# print(f"Mean DataLoader throughput: {np.mean(times):.0f} images/s")


# data_wait_times = []

# loader_iter = iter(train_dataloader)

# for i in range(512):
#     start = time.perf_counter()

#     images, labels = next(loader_iter)

#     elapsed = time.perf_counter() - start
#     data_wait_times.append(elapsed)

# print(
#     f"Mean batch retrieval: "
#     f"{sum(data_wait_times) / len(data_wait_times) * 1000:.2f} ms\n"
#     f"95th percentile: {1000 * np.percentile(data_wait_times, 95):.2f} ms\n"
#     f"99th percentile: {1000 * np.percentile(data_wait_times, 99):.2f} ms\n"
#     f"max: {1000 * np.max(data_wait_times):.2f} ms\n"
# )

# sys.exit()


device = "cuda" if torch.cuda.is_available() else "cpu"

torch.set_float32_matmul_precision("high")

config = ViTConfig(
    n_heads = 4,
    embd_size = 128,
    patch_size = 16,
    dropout = 0.2,
    num_encoders = 4,
    input_shape = (224, 224, 3),
    output_size = 1000,

)
model = ViT(config).to(device)
model = torch.compile(model) if device == "cuda" else model

epochs = 200
lr = 1e-3
weight_decay = 1e-2


optimizer = torch.optim.AdamW(
    params=model.parameters(), 
    lr=lr, 
    weight_decay=weight_decay,
    fused=True
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0= 100,
    T_mult = 1,
    eta_min = 1e-5
)
loss_fn = torch.nn.CrossEntropyLoss()



plot_metrics = {
    "train_losses": [],
    "val_losses": [],

    "train_accs": [],
    "val_accs": [],

    "train_epochs": [],
    "val_epochs": [],
}


# with profile(
#     activities=[
#         ProfilerActivity.CPU,
#         ProfilerActivity.CUDA,
#     ],
#     schedule=profiler_schedule,
#     on_trace_ready=tensorboard_trace_handler("runs/vit_profile"),
#     record_shapes=True,
#     profile_memory=True,
# ) as prof:

#     train_iter = iter(train_dataloader)

#     optimizer.zero_grad()
#     model.train()

#     batch_group_samples = 0
#     mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
#     std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

#     for i in range(20):
#         with record_function("DataLoader next"):
#             x_train, y_train = next(train_iter)

#         with record_function("H2D transfer"):
#             x_train = x_train.to(device, non_blocking=True)
#             y_train = y_train.to(device, non_blocking=True)

#         with record_function("GPU normalization"):
#             x_train = x_train.float().div_(255.0)
#             x_train.sub_(mean).div_(std)

#         x_train = x_train.contiguous(
#             memory_format=torch.channels_last
#         )

#         with record_function("MixUp"):
#             x_train, y_train = mixup(x_train, y_train)

#         batch_samples = len(x_train)


#         with record_function("Forward + loss"):
#             with torch.autocast(
#                 device,
#                 torch.bfloat16,
#                 enabled=True,
#             ):
#                 logits = model(x_train)
#                 loss = loss_fn(logits, y_train) * batch_samples

#         with record_function("Backward"):
#             loss.backward()

#         batch_group_samples += batch_samples

#         if (i + 1) % grad_accum_steps == 0:
#             with record_function("Gradient normalization"):
#                 for parameter in model.parameters():
#                     if parameter.grad is not None:
#                         parameter.grad.div_(batch_group_samples)

#             with record_function("Optimizer step"):
#                 optimizer.step()
#                 optimizer.zero_grad()
            
#             batch_group_samples = 0

#         prof.step()

#     # discard leftover gradients
#     optimizer.zero_grad()



In [ ]:



for epoch in range(epochs):
    current_lr = scheduler.get_last_lr()[0]
    pbar = tqdm(total=num_train_samples, desc = f"Epoch: {epoch+1}")

    
    train_model(
        model,
        train_dataloader,
        optimizer, loss_fn, device,
        pbar, plot_metrics,
        scheduler=scheduler,
        mixup=mixup, grad_accum_steps=grad_accum_steps
    )


    if (epoch == 0 or (epoch+1) % 5 == 0):
        evaluate_train_and_val(
            model, 
            clean_train_dataloader, val_dataloader,
            num_train_samples, num_val_samples,
            loss_fn, device, epoch,
            plot_metrics,
            train_eval_samples=10000
        )

        clear_output(wait=True)

        print(f"Epoch: {epoch+1} | "
              f"train_loss: {plot_metrics["train_losses"][-1] :.4f} | "
              f"train_acc: {plot_metrics["train_accs"][-1] :.2f} | "
              f"val_loss: {plot_metrics["val_losses"][-1] :.4f} | "
              f"val_acc: {plot_metrics["val_accs"][-1] :.2f} | "
              f"lr: {current_lr :.6f}\n"
        ) 


        plt.figure(figsize = (12, 6))

        plt.subplot(121)
        plt.plot(plot_metrics["train_epochs"], plot_metrics["train_losses"], label = "train loss")
        plt.plot(plot_metrics["val_epochs"], plot_metrics["val_losses"], label = "val loss")
        plt.xlabel("Epoch")
        plt.legend()

        plt.subplot(122)
        plt.plot(plot_metrics["val_epochs"], plot_metrics["train_accs"], label = "train acc")
        plt.plot(plot_metrics["val_epochs"], plot_metrics["val_accs"], label = "val acc")
        plt.xlabel("Epoch")
        plt.legend()

        plt.show()

Epoch: 1:  10%|█         | 132864/1281167 [00:26<02:39, 7219.79it/s]

4975 images/s



Epoch: 1:  21%|██        | 262656/1281167 [00:51<02:50, 5958.70it/s]

5771 images/s



Epoch: 1:  31%|███       | 393728/1281167 [01:17<02:14, 6617.06it/s]

5645 images/s



... (9 outputs left out)

5655 images/s



Epoch: 1:  92%|█████████▏| 1181440/1281167 [03:45<00:17, 5798.97it/s]

5591 images/s



Epoch: 1: 100%|██████████| 1281167/1281167 [04:13<00:00, 5054.35it/s] 
